# Basic Simulation — 32 nm Line/Space

This tutorial runs an end-to-end EUV lithography simulation:
1. **RCWA** compute mask diffraction orders (Ta absorber on Si)
2. **Aerial image** via Abbe source-point summation (NA 0.33, conventional illumination)
3. **Resist** exposure, PEB, and development
4. **CD extraction** from the resist profile

No GPU required — runs on CPU in ~15 seconds.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt

from euv.pipeline import SimulationConfig, run_simulation

In [ ]:
cfg = SimulationConfig(
    period_nm=64.0,      # 64 nm pitch
    line_width_nm=32.0,  # 32 nm lines
    dose_mj_cm2=20.0,    # 20 mJ/cm² dose
    na=0.33,             # numerical aperture
    sigma=0.8,           # partial coherence
    grid=256,            # simulation grid
    n_rcwa_orders=21,    # RCWA Fourier orders
)

result = run_simulation(cfg)
print(f"CD = {result.cd_nm:.2f} nm")
print(f"NILS = {result.nils_value:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

im1 = ax1.imshow(result.aerial_image.cpu().numpy(), cmap='hot', aspect='equal')
ax1.set_title('Aerial Image (intensity)')
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(result.resist_profile.cpu().numpy(), cmap='gray', aspect='equal')
ax2.set_title(f'Resist Profile — CD = {result.cd_nm:.1f} nm')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

## Interpret the results

- **CD ≈ 29.5 nm** for a 32 nm design line — the resist bias narrows the line slightly.
- **Aerial image** shows the diffracted intensity pattern at wafer.
- **Resist profile** shows cleared (dark) vs remaining (bright) regions.
- The absorber (Ta) reflects very little at 13.5 nm (~0.06 %), as expected.